In [ ]:
import xarray as xr
import os

SEASONAL_DIR = "/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/Data for Dev/seasonal_sequences"
YIELD_DIR    = "/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/rounded_farm"
OUTPUT_DIR   = "/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/Data for Dev/final_with_yield"

os.makedirs(OUTPUT_DIR, exist_ok=True)

YEARS = range(1991, 2024)


In [ ]:
for year in YEARS:
    print(f"🔄 Merging yield for {year}")

    # -------------------------------------------------
    # Load seasonal features
    # -------------------------------------------------
    seasonal_ds = xr.open_dataset(
        os.path.join(SEASONAL_DIR, f"seasonal_features_{year}.nc")
    )

    # -------------------------------------------------
    # Load yield
    # -------------------------------------------------
    yield_ds = xr.open_dataset(
        os.path.join(YIELD_DIR, f"wheat_farm_{year}.nc")
    )

    # -------------------------------------------------
    # Identify yield variable safely
    # -------------------------------------------------
    if "H_wheat_dot_hat" in yield_ds.data_vars:
        yvar = "H_wheat_dot_hat"
    elif "wheat_yield" in yield_ds.data_vars:
        yvar = "wheat_yield"
    else:
        raise ValueError(f"No yield variable found in {year}")

    # -------------------------------------------------
    # Ensure yield has no time dimension
    # (annual → spatial only)
    # -------------------------------------------------
    if "time" in yield_ds[yvar].dims:
        yield_da = yield_ds[yvar].squeeze("time", drop=True)
    else:
        yield_da = yield_ds[yvar]

    # -------------------------------------------------
    # Merge seasonal + yield
    # -------------------------------------------------
    final_ds = xr.merge(
        [seasonal_ds, yield_da.rename("yield")],
        compat="no_conflicts"
    )

    # -------------------------------------------------
    # Metadata
    # -------------------------------------------------
    final_ds.attrs.update({
        "title": "Seasonal Climate, Drought Indices, and Wheat Yield",
        "year": year,
        "season": "May–October",
        "yield_type": "Annual wheat yield",
        "created_by": "xarray seasonal-yield merge"
    })

    # -------------------------------------------------
    # Save
    # -------------------------------------------------
    out_file = os.path.join(OUTPUT_DIR, f"final_dataset_{year}.nc")
    final_ds.to_netcdf(out_file)

    print(f"✅ Saved {out_file}")

print("🎉 Yield merged successfully for all years!")


In [ ]:
ds = xr.open_dataset("/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/Data for Dev/final_with_yield/final_dataset_1991.nc")
print(ds)

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path

# === Load the shapefile ===
shapefile_path = r'/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Eastern_Aus_Bunbury/Eastern_Aus_Bunbury.shp'
qld_shape = gpd.read_file(shapefile_path).to_crs("EPSG:4326")

# === Open the SPI dataset ===
nc_path = Path(r'/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/Data for Dev/final_with_yield/final_dataset_1991.nc')
year = nc_path.stem.split('_')[1]

ds = xr.open_dataset(nc_path)

# === Select SPI variable ONCE ===
var_name = 'spi_3'          # change here: spi_3, spi_6, etc.
monthly_rain = ds[var_name]

# === Convert variable name to display label ===
spi_label = var_name.replace('_', '-').upper()   # spi_1 → SPI-1

# === Fix metadata so colorbar label is correct ===
monthly_rain.attrs.pop('units', None)
monthly_rain.attrs['long_name'] = spi_label

# === Create figure and axes ===
fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(20, 12))
axes = axes.flatten()

# === Plot each month ===
for i, month in enumerate(monthly_rain.time):
    ax = axes[i]
    data = monthly_rain.sel(time=month)

    data.plot(
        ax=ax,
        cmap='RdBu',
        add_colorbar=True
    )

    # Overlay region boundaries
    qld_shape.boundary.plot(
        ax=ax,
        color='black',
        linewidth=0.5,
        alpha=0.7
    )

    # Titles and formatting
    ax.set_title(month.dt.strftime('%B').item(), fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('')

# === Remove unused subplots ===
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# === Figure title (automatic) ===
plt.suptitle(
    f'Monthly {spi_label} for {year} in WheatBelt',
    fontsize=16,
    y=0.98
)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

# Load one merged file
ds = xr.open_dataset("/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/Data for Dev/final_with_yield/final_dataset_1995.nc")

# Plot Yield
plt.figure(figsize=(10, 5))
ds['yield'].plot(cmap='viridis')
plt.title("Yield Map (Check if this looks distorted)")
plt.show()

In [ ]:
ds = xr.open_dataset("/content/drive/MyDrive/Research Works - Development/Work 2 - SPI SPEI/Data/raster/rounded_farm/wheat_farm_1991.nc")
print(ds)